# Recomendador de Artículos Similares por Perfil de Tópicos

Este notebook carga el modelo híbrido de BERTopic y las distribuciones de probabilidad de los tópicos para buscar los 10 artículos más cercanos temáticamente a un `doc_id` dado, utilizando la distancia coseno.

In [23]:
# 1. Parche de compatibilidad obligatorio para BERTopic
import coverage.types as ct
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

# 2. Imports generales
import pandas as pd
import numpy as np
from pathlib import Path
from bertopic import BERTopic
from sklearn.metrics.pairwise import cosine_similarity
from umap import UMAP
import plotly.graph_objects as go
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings("ignore")

# 3. Configuración de Rutas
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_lematizado.csv"
MODEL_PATH = PROJECT_ROOT / "outputs" / "bertopic_hibrido" / "bertopic_hibrido_model"

# 4. Carga de Datos
df = pd.read_csv(DATA_PROCESSED)
df["doc_id"] = df.index

print(f"Corpus cargado: {len(df):,} documentos listos para analizar.")

Corpus cargado: 3,556 documentos listos para analizar.


In [24]:
# ==============================================================================
# CARGA DEL MODELO Y DISTRIBUCIONES DE PROBABILIDAD DE TÓPICOS
# ==============================================================================

# 1. Cargar el modelo BERTopic preentrenado
print(f"Cargando modelo desde: {MODEL_PATH.name}...")
topic_model = BERTopic.load(MODEL_PATH)

# 2. Calcular la distribución probabilística suave de tópicos por artículo
print("Calculando distribuciones continuas de tópicos para todos los documentos...")
textos_docs = df["document_text_final"].fillna("").astype(str).tolist()
doc_topic_dist, _ = topic_model.approximate_distribution(textos_docs, use_embedding_model=False)

print(f"Matriz de distribución generada. Shape: {doc_topic_dist.shape}")

Cargando modelo desde: bertopic_hibrido_model...
Calculando distribuciones continuas de tópicos para todos los documentos...
Matriz de distribución generada. Shape: (3556, 43)


In [25]:
# ==============================================================================
# FUNCIÓN DE SIMILITUD Y AUDITORÍA DE ETIQUETAS (CON TAGLINE)
# ==============================================================================

def recomendar_similares_por_topico(target_doc_id, top_n=10):
    """
    Toma un doc_id, busca su perfil de tópicos y devuelve los top_n artículos 
    con mayor similitud coseno, analizando el estado de sus etiquetas y mostrando la bajada.
    """
    if target_doc_id not in df["doc_id"].values:
        return f"Error: doc_id {target_doc_id} no encontrado en el corpus."
    
    # 1. Extraer vector de distribución temática del documento objetivo
    target_vector = doc_topic_dist[target_doc_id].reshape(1, -1)
    
    # 2. Calcular Similitud Coseno contra todo el corpus
    similitudes = cosine_similarity(target_vector, doc_topic_dist).flatten()
    
    # 3. Ordenar índices de mayor a menor
    indices_similares = similitudes.argsort()[::-1][:top_n + 1]
    
    # Excluir el documento objetivo de la lista de resultados
    indices_similares = [idx for idx in indices_similares if idx != target_doc_id][:top_n]
    
    # 4. Construir tabla de resultados con diagnóstico de etiquetas y tagline
    resultados = []
    for rank, idx in enumerate(indices_similares, start=1):
        row = df.iloc[idx]
        
        # Validar existencia de la etiqueta
        etiqueta_raw = row.get("post_tag", np.nan)
        tiene_etiqueta = pd.notna(etiqueta_raw) and str(etiqueta_raw).strip() != ""
        
        # Limpiar tagline
        tagline_raw = str(row.get("tagline", "")).strip()
        tagline_limpio = tagline_raw if tagline_raw and tagline_raw.lower() != "nan" else "Sin bajada"
        
        resultados.append({
            "Rank": rank,
            "doc_id": row["doc_id"],
            "Similitud_Coseno": round(float(similitudes[idx]), 4),
            "Título": row["title_text"],
            "Tagline": tagline_limpio,
            #"Tiene_Etiqueta": tiene_etiqueta,
            "Etiquetas": str(etiqueta_raw).strip() if tiene_etiqueta else "Sin etiquetas"
        })
        
    df_resultados = pd.DataFrame(resultados)
    
    # Información del documento consultado
    doc_origen = df.iloc[target_doc_id]
    
    etiqueta_origen = doc_origen.get("post_tag", np.nan)
    origen_tiene_tag = pd.notna(etiqueta_origen) and str(etiqueta_origen).strip() != ""
    
    tagline_origen_raw = str(doc_origen.get("tagline", "")).strip()
    tagline_origen = tagline_origen_raw if tagline_origen_raw and tagline_origen_raw.lower() != "nan" else "Sin bajada"
    
    print("=" * 100)
    print(f"[ID: {target_doc_id}]")
    print(f"Título:  {doc_origen['title_text']}")
    print(f"Tagline: {tagline_origen}")
    print(f"Tags:    {etiqueta_origen if origen_tiene_tag else 'Sin etiquetas'}")
    print("=" * 100)
    print(f"\nTOP {top_n} NOTAS MÁS SIMILARES POR PERFIL DE TÓPICOS:\n")
    
    return df_resultados

In [26]:
# ==============================================================================
# 2. CÁLCULO SILENCIOSO DE LAS DOS DIMENSIONES (SEMÁNTICA Y TEMÁTICA)
# ==============================================================================

# A. Espacio Semántico (Embeddings de texto puro original)
if "umap_sem_x" not in df.columns:
    print("Preparando Espacio Semántico (Textos puros)...")
    if 'embeddings_naturales' not in globals():
        emb_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
        # Usamos document_text_original para no perder contexto gramatical
        textos_originales = df["document_text_original"].fillna("").astype(str).tolist()
        embeddings_naturales = emb_model.encode(textos_originales, show_progress_bar=False)
        
    reducer_sem = UMAP(n_neighbors=15, n_components=2, metric='cosine', random_state=42)
    coords_sem = reducer_sem.fit_transform(embeddings_naturales)
    df["umap_sem_x"] = coords_sem[:, 0]
    df["umap_sem_y"] = coords_sem[:, 1]

# B. Espacio Temático (Probabilidades de Tópicos BERTopic)
if "umap_tem_x" not in df.columns:
    print("Preparando Espacio Temático (Distribución BERTopic)...")
    reducer_tem = UMAP(n_neighbors=15, n_components=2, metric='cosine', random_state=42)
    coords_tem = reducer_tem.fit_transform(doc_topic_dist)
    df["umap_tem_x"] = coords_tem[:, 0]
    df["umap_tem_y"] = coords_tem[:, 1]


Preparando Espacio Semántico (Textos puros)...
Preparando Espacio Temático (Distribución BERTopic)...


In [33]:
import plotly.express as px
from umap import UMAP
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ==============================================================================
# 1. PREPARAR PALABRAS CLAVE DE TÓPICOS PARA LOS COLORES
# ==============================================================================
# 1. Obtener los IDs válidos excluyendo outliers (-1)
topic_info = topic_model.get_topic_info()
valid_topics = [t for t in topic_info["Topic"] if t != -1]

# 2. Mapeo de IDs a sus palabras clave lematizadas (ej: "0_eleccion_voto_candidato")
topic_words_dict = {}
for t_id in valid_topics:
    words_weights = topic_model.get_topic(t_id)
    if words_weights:
        topic_words_dict[t_id] = f"{t_id}_" + "_".join([w for w, _ in words_weights[:3]])
    else:
        topic_words_dict[t_id] = f"Topic_{t_id}"

# 3. Asignar el tópico dominante a cada documento si la columna no existe
if "topic" not in df.columns:
    dominant_indices = np.argmax(doc_topic_dist, axis=1)
    df["topic"] = [valid_topics[i] for i in dominant_indices]

# 4. Formatear la etiqueta para la leyenda y hover
df["topic_name"] = df["topic"].map(topic_words_dict)
df["topic_label"] = df["topic_name"].fillna("Sin tópico")

# Asegurar tagline formateado sin tocar las demás columnas
if "tagline" in df.columns:
    df["tagline"] = df["tagline"].fillna("").astype(str).replace(["nan", "None"], "")
else:
    df["tagline"] = ""

# 5. Ordenar la leyenda numéricamente
topics_ordenados = sorted(df["topic"].unique())
categorias_ordenadas = [topic_words_dict.get(t, f"Topic_{t}") for t in topics_ordenados]
df["topic_label"] = pd.Categorical(df["topic_label"], categories=categorias_ordenadas, ordered=True)

# ==============================================================================
# 2. FUNCIÓN DE VISUALIZACIÓN MEJORADA (Colores + Formas)
# ==============================================================================
def visualizar_constelacion_color(target_doc_id, df_similares, col_x, col_y, titulo_grafico):
    recomendados_ids = df_similares["doc_id"].tolist()
    
    # Dividir los datos
    df_resto = df[~df["doc_id"].isin([target_doc_id] + recomendados_ids)]
    df_rec = df.loc[recomendados_ids].copy()
    df_rec["Rank"] = df_rec["doc_id"].map(df_similares.set_index("doc_id")["Rank"])
    df_target = df.loc[[target_doc_id]]
    
    # Capa 1: Fondo coloreado por Tópicos (usando Plotly Express para gestionar la paleta)
    fig = px.scatter(
        df_resto, x=col_x, y=col_y, color='topic_label',
        color_discrete_sequence=px.colors.qualitative.Alphabet,
        custom_data=['topic_label', 'title_text', 'tagline']
    )
    
    # Hacer los puntos del fondo más chicos y transparentes
    fig.update_traces(
        marker=dict(size=6, opacity=0.5),
        hovertemplate="<b>%{customdata[0]}</b><br><b>%{customdata[1]}</b><br><i>%{customdata[2]}</i><extra></extra>"
    )
    
    # Capa 2: Las Recomendaciones (Círculos)
    fig.add_trace(go.Scatter(
        x=df_rec[col_x], y=df_rec[col_y], mode="markers", name="Top 10 Recomendados",
        marker=dict(symbol="circle", size=16, line=dict(width=3, color="#1f77b4")),
        hovertemplate="<b>Recomendación #%{customdata[1]}</b><br><b>Tópico:</b> %{customdata[2]}<br><b>%{customdata[0]}</b><br><i>%{customdata[3]}</i><extra></extra>", 
        customdata=df_rec[["title_text", "Rank", "topic_label", "tagline"]]
    ))
    
    # Capa 3: El artículo consultado / Origen (Diamante Rojo)
    fig.add_trace(go.Scatter(
        x=df_target[col_x], y=df_target[col_y], mode="markers", name="Artículo Consultado",
        marker=dict(color="#d62728", size=20, symbol="diamond", line=dict(width=2, color="black")),
        hovertemplate="<b>ARTÍCULO ORIGINAL</b><br><b>Tópico:</b> %{customdata[1]}<br><b>%{customdata[0]}</b><br><i>%{customdata[2]}</i><extra></extra>", 
        customdata=df_target[["title_text", "topic_label", "tagline"]]
    ))
    
    fig.update_layout(
        title=f"<b>{titulo_grafico}</b>",
        plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#fff', height=800,
        hoverlabel=dict(bgcolor="white", font_size=14),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
        legend=dict(title="<b>Referencias y Tópicos</b>", yanchor="top", y=1, xanchor="left", x=1.02)
    )
    return fig

In [34]:
DOC_ID_A_CONSULTAR = 150
df_similares = recomendar_similares_por_topico(DOC_ID_A_CONSULTAR, top_n=10)
display(df_similares)

# Gráfico 1: El texto crudo
fig_semantico = visualizar_constelacion_color(
    DOC_ID_A_CONSULTAR, df_similares, 
    col_x="umap_sem_x", col_y="umap_sem_y", 
    titulo_grafico="Constelación SEMÁNTICA (Ubicación por texto del articulo)"
)
fig_semantico.show()

# Gráfico 2: La receta de temas
fig_tematico = visualizar_constelacion_color(
    DOC_ID_A_CONSULTAR, df_similares, 
    col_x="umap_tem_x", col_y="umap_tem_y", 
    titulo_grafico="Constelación TEMÁTICA (Ubicación por topico)"
)
fig_tematico.show()

[ID: 150]
Título:  Messi vació las calles
Tagline: Argentina vs Nigeria
Tags:    Sin etiquetas

TOP 10 NOTAS MÁS SIMILARES POR PERFIL DE TÓPICOS:



,Rank,doc_id,Similitud_Coseno,Título,Tagline,Etiquetas
0,1,2348,0.9793,El poder de la camiseta,Argentina campeón,Sin etiquetas
1,2,1724,0.9746,"Diego, Leo, Trinche",Un crack sin Youtube,Sin etiquetas
2,3,2708,0.9746,El jugador que nos habíamos perdido está acá,El último Mundial de Messi,Fútbol; Messi; Mundial de fútbol; Qatar 2022
3,4,3130,0.9744,Nos habíamos negado a la belleza,El último baile de Messi y Di María juntos,Di María; Fútbol; Lionel Messi; Selección Arge...
4,5,147,0.9734,Los mandamientos de Sabella,Mundial 2014,Alejandro Sabella; Brasil; Internacional; Mund...
5,6,2752,0.9727,La batalla de Lusail,Argentina - Países Bajos,Fútbol; Messi; Mundial de fútbol; Qatar 2022; ...
6,7,169,0.9714,Cruzamos el Rubicón,Argentina vs Bélgica,Alejandro Sabella; Brasil; Internacional; Mund...
7,8,1520,0.9712,La maldición de depender del milagro,Mundial 2018,Sin etiquetas
8,9,3383,0.9710,El prócer tardío,"Ángel Di María, del estigma al heroísmo",Di María; Fútbol; Messi; Rosario; Rosariocroni...
9,10,1050,0.9694,Tenemos histeria porque no tenemos fútbol,La Selección y el mundial,Sin etiquetas
